# Real 2024 Vecchia: BaseLocal vs NearLocal Hybrid

This notebook reuses the real-data loading pattern from `gpu_dw_fit_complex.ipynb`, but fits Vecchia models instead of Debiased Whittle.

Comparison target for 2024 July first 5 days:

1. `BaseLocal_A20_B20_C20`: original same-location/local structure baseline.
2. `NearLocal_L16F02_C12F02_Op0p063`: shared regular-grid ordering, actual-coordinate likelihood, hybrid lag sets with lag-1 east offset 0.063.
3. `NearLocal_L16F02_C12F02_Op0p126`: same, lag-1 east offset 0.126.

Because this is real data, there is no true parameter vector. The comparison focuses on profile loss, fitted physical parameters, timing, and Godambe standard errors / relative SE-to-estimate diagnostics.

In [1]:
import os
import sys
import time
import io
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn import Parameter

AMAREL_SRC = "/home/jl2815/tco"
LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
_src = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, _src)

from GEMS_TCO import configuration as config
from GEMS_TCO import kernels_vecchia
from GEMS_TCO.data_loader import load_data_dynamic_processed

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

print("SRC:", _src)
print("DEVICE:", DEVICE)

SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
DEVICE: cpu


## Settings

`keep_ori=True` restores the real source locations for the Vecchia likelihood. The regular grid is used for shared ordering and the base neighbor scaffold only.

In [2]:
YEAR = "2024"
MONTH = 7
DAYS_LIST = list(range(5))  # 0-based: July 1-5
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

LAT_LON_RESOLUTION = [1, 1]
DELTA_LAT_BASE = 0.044
DELTA_LON_BASE = 0.063
MM_COND_NUMBER = 100
SMOOTH = 0.5
NHEADS = 0
DAILY_STRIDE = 2

LBFGS_LR = 1.0
LBFGS_MAX_STEPS = 5
LBFGS_MAX_EVAL = 20
LBFGS_HISTORY_SIZE = 10
GRAD_TOL = 1e-5
SUPPRESS_FIT_PRINTS = True

COMPUTE_GODAMBE = True
HESSIAN_EPS = 1e-4
SCORE_EPS = 1e-5
H_RIDGE_SCALE = 1e-6
GODAMBE_J_METHOD = "block"  # block, per_unit_centered, per_unit_uncentered
GODAMBE_BLOCK_LAT_WIDTH = 0.50
GODAMBE_BLOCK_LON_WIDTH = 0.50
GODAMBE_BLOCK_TIME_WIDTH = 2.0

OUT_DIR = Path("/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log")
OUT_PREFIX = "real_vecc_2024_5day_nearlocal_vs_base_050226"
SAVE_CSV = True

INIT_PHYSICAL = {
    "sigmasq": 10.0,
    "range_lat": 0.30,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.05,
    "advec_lon": -0.10,
    "nugget": 2.5,
}

print("days:", DAYS_LIST)
print("mm_cond_number:", MM_COND_NUMBER)
print("Godambe:", COMPUTE_GODAMBE, GODAMBE_J_METHOD)

days: [0, 1, 2, 3, 4]
mm_cond_number: 100
Godambe: True block


## Model Specs

In [3]:
def offset_tag(x):
    sign = "p" if x >= 0 else "m"
    return sign + f"{abs(float(x)):.3f}".replace(".", "p")

def base_local_spec():
    return {
        "name": "BaseLocal_A20_B20_C20",
        "group": "local_baseline",
        "kernel": "std",
        "limit_A": 20,
        "limit_B": 20,
        "limit_C": 20,
        "lag1_local_count": 20,
        "lag1_fresh_count": 0,
        "lag2_local_count": 20,
        "lag2_fresh_count": 0,
        "pred_lag1_lon_offset": 0.0,
        "pred_lag2_lon_offset": 0.0,
        "pred_lag1_cells": 0.0,
        "pred_lag2_cells": 0.0,
        "total_conditioning": 20 + 1 + 20 + 1 + 20,
        "allocation": "A20; B=same+local20; C=same+local20",
        "description": "original same-location local-structure baseline: 20/20/20",
    }

def nearlocal_spec(offset):
    tag = offset_tag(offset)
    return {
        "name": f"NearLocal_L16F02_C12F02_O{tag}",
        "group": "near_local_hybrid",
        "kernel": "hybrid_fresh",
        "limit_A": 20,
        "limit_B": 16,
        "limit_C": 12,
        "lag1_local_count": 16,
        "lag1_fresh_count": 2,
        "lag2_local_count": 12,
        "lag2_fresh_count": 2,
        "pred_lag1_lon_offset": float(offset),
        "pred_lag2_lon_offset": float(2.0 * offset),
        "pred_lag1_cells": float(offset / DELTA_LON_BASE),
        "pred_lag2_cells": float(2.0 * offset / DELTA_LON_BASE),
        "total_conditioning": 20 + 1 + 16 + 2 + 1 + 12 + 2,
        "allocation": f"A20; B=same+local16+fresh2; C=same+local12+fresh2; lag1 offset={offset:.3f}",
        "description": "near-local hybrid reference: mostly local with a small shifted/fresh component",
    }

MODEL_SPECS = {
    "local_baseline::BaseLocal_A20_B20_C20": base_local_spec(),
    "near_local_hybrid::NearLocal_L16F02_C12F02_Op0p063": nearlocal_spec(0.063),
    "near_local_hybrid::NearLocal_L16F02_C12F02_Op0p126": nearlocal_spec(0.126),
}

spec_df = pd.DataFrame(MODEL_SPECS).T
print("models:", len(MODEL_SPECS), "fits:", len(MODEL_SPECS) * len(DAYS_LIST))
display(spec_df[["group", "kernel", "limit_A", "lag1_local_count", "lag1_fresh_count", "lag2_local_count", "lag2_fresh_count", "pred_lag1_lon_offset", "total_conditioning", "allocation"]])

models: 3 fits: 15


,group,kernel,limit_A,lag1_local_count,lag1_fresh_count,lag2_local_count,lag2_fresh_count,pred_lag1_lon_offset,total_conditioning,allocation
local_baseline::BaseLocal_A20_B20_C20,local_baseline,std,20,20,0,20,0,0.0,62,A20; B=same+local20; C=same+local20
near_local_hybrid::NearLocal_L16F02_C12F02_Op0p063,near_local_hybrid,hybrid_fresh,20,16,2,12,2,0.063,54,A20; B=same+local16+fresh2; C=same+local12+fre...
near_local_hybrid::NearLocal_L16F02_C12F02_Op0p126,near_local_hybrid,hybrid_fresh,20,16,2,12,2,0.126,54,A20; B=same+local16+fresh2; C=same+local12+fre...


## Load 2024 July Real Data

This mirrors `gpu_dw_fit_complex.ipynb`, except `is_whittle=False` so the shared max-min ordering/NNS map is generated. `keep_ori=True` means the likelihood sees actual source coordinates, while `nns_map` remains the regular-grid ordering scaffold.

In [4]:
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=LAT_LON_RESOLUTION,
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

if ord_mm is None or nns_map is None:
    raise RuntimeError("Vecchia ordering failed: ord_mm/nns_map is None")

key_idx = sorted(df_map)
print("n hourly slots:", len(key_idx))
print("monthly_mean:", monthly_mean)
print("ord length:", len(ord_mm), "nns shape/len:", getattr(nns_map, "shape", len(nns_map)))
print("first key:", key_idx[0], "last key:", key_idx[-1])

first_df_ord = df_map[key_idx[0]].iloc[ord_mm].reset_index(drop=True)
ordered_base_coords_np = first_df_ord[["Latitude", "Longitude"]].to_numpy(dtype=np.float64)
print("ordered_base_coords_np:", ordered_base_coords_np.shape)

--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
n hourly slots: 248
monthly_mean: 257.9726104252314
ord length: 18126 nns shape/len: (18126, 100)
first key: 2024_07_y24m07day01_hm00:53 last key: 2024_07_y24m07day31_hm07:48
ordered_base_coords_np: (18126, 2)


In [5]:
daily_hourly_maps_vecc = {}
daily_aggregated_tensors_vecc = {}
selected_key_map = {}

for day_idx in DAYS_LIST:
    hour_indices = [day_idx * 8, (day_idx + 1) * 8]
    selected_key_map[day_idx] = key_idx[hour_indices[0]:hour_indices[1]]
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map,
        monthly_mean,
        hour_indices,
        ord_mm=ord_mm,
        dtype=DTYPE,
        keep_ori=True,
    )
    daily_hourly_maps_vecc[day_idx] = day_hourly_map
    daily_aggregated_tensors_vecc[day_idx] = day_aggregated_tensor

load_rows = []
for day_idx in DAYS_LIST:
    maps = daily_hourly_maps_vecc[day_idx]
    n_valid = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in maps.values())
    n_total = sum(len(v) for v in maps.values())
    load_rows.append({
        "day_idx": day_idx,
        "day": f"{YEAR}-{MONTH:02d}-{day_idx + 1:02d}",
        "n_time_slots": len(maps),
        "n_rows_total": n_total,
        "n_valid_o3": n_valid,
        "valid_rate": n_valid / max(n_total, 1),
        "first_slot": selected_key_map[day_idx][0] if selected_key_map[day_idx] else None,
        "last_slot": selected_key_map[day_idx][-1] if selected_key_map[day_idx] else None,
    })
load_summary = pd.DataFrame(load_rows)
display(load_summary)

,day_idx,day,n_time_slots,n_rows_total,n_valid_o3,valid_rate,first_slot,last_slot
0,0,2024-07-01,8,145008,131428,0.906350,2024_07_y24m07day01_hm00:53,2024_07_y24m07day01_hm07:48
1,1,2024-07-02,8,145008,140191,0.966781,2024_07_y24m07day02_hm00:53,2024_07_y24m07day02_hm07:48
2,2,2024-07-03,8,145008,140352,0.967891,2024_07_y24m07day03_hm00:53,2024_07_y24m07day03_hm07:48
3,3,2024-07-04,8,145008,139272,0.960444,2024_07_y24m07day04_hm00:53,2024_07_y24m07day04_hm07:48
4,4,2024-07-05,8,145008,142684,0.983973,2024_07_y24m07day05_hm00:53,2024_07_y24m07day05_hm07:48


## Hybrid Kernel

This is the same near-local/fresh hybrid construction used in the low-resolution simulations: same-location temporal anchor, local neighbors around the current base cell, and a small fresh set around the east-shifted center. The likelihood itself uses the actual source coordinates in `input_map`.

In [6]:
class fit_vecchia_lbfgs_fresh_hybrid(kernels_vecchia.fit_vecchia_lbfgs):
    """Vecchia kernel mixing local lag neighbors with shifted-center fresh lag neighbors."""

    def __init__(
        self,
        smooth,
        input_map,
        nns_map,
        mm_cond_number,
        nheads,
        limit_A=20,
        limit_B=16,
        limit_C=12,
        daily_stride=2,
        spatial_coords=None,
        lag1_lon_offset=0.063,
        lag1_fresh_count=2,
        lag2_fresh_count=2,
    ):
        super().__init__(
            smooth,
            input_map,
            nns_map,
            mm_cond_number,
            nheads,
            limit_A=limit_A,
            limit_B=limit_B,
            limit_C=limit_C,
            daily_stride=daily_stride,
        )
        self.spatial_coords = spatial_coords
        self.lag1_lon_offset = float(abs(lag1_lon_offset))
        self.lag1_fresh_count = int(lag1_fresh_count)
        self.lag2_fresh_count = int(lag2_fresh_count)

    def _spatial_coords_np(self, n_points):
        if self.spatial_coords is not None:
            coords_np = np.asarray(self.spatial_coords[:n_points], dtype=np.float64)
        else:
            all_data = [torch.from_numpy(d) if isinstance(d, np.ndarray) else d for d in self.input_map.values()]
            coords_np = all_data[0][:n_points, :2].detach().cpu().numpy().astype(np.float64)
        coords_np = coords_np.copy()
        nan_mask = np.isnan(coords_np).any(axis=1)
        coords_np[nan_mask] = np.array([0.0, 1000.0])
        return coords_np

    def _build_shift_lookup(self, n_points, multiplier):
        from sklearn.neighbors import BallTree

        coords_np = self._spatial_coords_np(n_points)
        tree = BallTree(np.radians(coords_np), metric="haversine")
        lats = coords_np[:, 0]
        lons = coords_np[:, 1]
        valid = ~np.isnan(coords_np).any(axis=1)
        lon_min = float(np.nanmin(lons[valid]))
        lon_max = float(np.nanmax(lons[valid]))
        base_ids = np.arange(n_points, dtype=np.int64)

        target_lons = lons + multiplier * self.lag1_lon_offset
        outside = (~valid) | (target_lons < lon_min) | (target_lons > lon_max)
        query = np.column_stack([np.radians(lats), np.radians(target_lons)])
        _, idx = tree.query(query, k=1)
        lookup = idx.flatten().astype(np.int64)
        lookup[outside] = base_ids[outside]
        return lookup

    def precompute_conditioning_sets(self):
        limit_A = int(self.limit_A)
        lag1_local = int(self.limit_B)
        lag2_local = int(self.limit_C)
        lag1_fresh = int(self.lag1_fresh_count)
        lag2_fresh = int(self.lag2_fresh_count)
        daily_stride = int(self.daily_stride)

        max_dim_A = limit_A
        max_dim_AB = limit_A + 1 + lag1_local + lag1_fresh
        max_dim_ABC = max_dim_AB + 1 + lag2_local + lag2_fresh

        all_data_list = [torch.from_numpy(d) if isinstance(d, np.ndarray) else d for d in self.input_map.values()]
        Real_Data = torch.cat(all_data_list, dim=0).to(self.device, dtype=torch.float32)
        n_real, num_cols = Real_Data.shape

        is_nan_real = torch.isnan(Real_Data[:, 2])
        valid_lats = Real_Data[~is_nan_real, 0]
        self.lat_mean_val = valid_lats.mean().item() if valid_lats.numel() > 0 else Real_Data[:, 0].mean().item()
        is_nan_mask_np = is_nan_real.cpu().numpy()

        n_dummies = max_dim_ABC
        dummy_block = torch.zeros((n_dummies, num_cols), device=self.device, dtype=torch.float32)
        for k in range(n_dummies):
            dummy_block[k, 0] = (k + 1) * 1e8
            dummy_block[k, 1] = (k + 1) * 1e8
            dummy_block[k, 3] = (k + 1) * 1e8
        Full_Data = torch.cat([Real_Data, dummy_block], dim=0)
        dummy_start = n_real
        is_nan_mask_np = np.append(is_nan_mask_np, np.zeros(n_dummies, dtype=bool))

        key_list = list(self.input_map.keys())
        day_lengths = [len(d) for d in all_data_list]
        cumulative_len = np.cumsum([0] + day_lengths)
        n_time_steps = len(key_list)
        use_set_C = daily_stride < n_time_steps

        n_pts_per_day = day_lengths[0]
        lag1_center = self._build_shift_lookup(n_pts_per_day, multiplier=1.0)
        lag2_center = self._build_shift_lookup(n_pts_per_day, multiplier=2.0)

        heads_indices = []
        batch_list_A = []
        batch_list_AB = []
        batch_list_ABC = []

        def add_valid_neighbors(indices_to_check, current_indices, cap):
            count = 0
            seen = set(current_indices)
            for idx in indices_to_check:
                if count >= cap:
                    break
                idx = int(idx)
                if idx < 0 or idx in seen:
                    continue
                if idx < len(is_nan_mask_np) and not is_nan_mask_np[idx]:
                    current_indices.append(idx)
                    seen.add(idx)
                    count += 1

        for time_idx, key in enumerate(key_list):
            day_len = day_lengths[time_idx]
            offset = cumulative_len[time_idx]

            for local_idx in range(min(day_len, self.nheads)):
                idx = offset + local_idx
                if not is_nan_mask_np[idx]:
                    heads_indices.append(idx)
            if self.nheads >= day_len:
                continue

            for local_idx in range(self.nheads, day_len):
                target_idx = offset + local_idx
                if is_nan_mask_np[target_idx]:
                    continue

                current_indices = []
                nbs_current = self.nns_map[local_idx] if local_idx < len(self.nns_map) else np.array([], dtype=np.int64)
                add_valid_neighbors((offset + nbs_current).tolist(), current_indices, cap=limit_A)

                has_B = time_idx > 0
                has_C = time_idx >= daily_stride

                if has_B:
                    prev_off = cumulative_len[time_idx - 1]
                    prev_len = day_lengths[time_idx - 1]
                    if local_idx < prev_len:
                        add_valid_neighbors([prev_off + local_idx], current_indices, cap=1)

                    local_candidates = [prev_off + int(v) for v in nbs_current if int(v) < prev_len and int(v) != local_idx]
                    add_valid_neighbors(local_candidates, current_indices, cap=lag1_local)

                    center_B = int(lag1_center[local_idx]) if local_idx < len(lag1_center) else local_idx
                    if center_B >= prev_len:
                        center_B = local_idx
                    nbs_B = self.nns_map[center_B] if center_B < len(self.nns_map) else np.array([], dtype=np.int64)
                    fresh_candidates = [prev_off + center_B] + [prev_off + int(v) for v in nbs_B if int(v) < prev_len and int(v) != local_idx]
                    add_valid_neighbors(fresh_candidates, current_indices, cap=lag1_fresh)

                if has_C:
                    pd_idx = time_idx - daily_stride
                    pd_off = cumulative_len[pd_idx]
                    pd_len = day_lengths[pd_idx]
                    if local_idx < pd_len:
                        add_valid_neighbors([pd_off + local_idx], current_indices, cap=1)

                    local_candidates = [pd_off + int(v) for v in nbs_current if int(v) < pd_len and int(v) != local_idx]
                    add_valid_neighbors(local_candidates, current_indices, cap=lag2_local)

                    center_C = int(lag2_center[local_idx]) if local_idx < len(lag2_center) else local_idx
                    if center_C >= pd_len:
                        center_C = local_idx
                    nbs_C = self.nns_map[center_C] if center_C < len(self.nns_map) else np.array([], dtype=np.int64)
                    fresh_candidates = [pd_off + center_C] + [pd_off + int(v) for v in nbs_C if int(v) < pd_len and int(v) != local_idx]
                    add_valid_neighbors(fresh_candidates, current_indices, cap=lag2_fresh)

                if has_C:
                    max_d, target_list = max_dim_ABC, batch_list_ABC
                elif has_B:
                    max_d, target_list = max_dim_AB, batch_list_AB
                else:
                    max_d, target_list = max_dim_A, batch_list_A

                n_valid = len(current_indices)
                if n_valid < max_d:
                    row = [dummy_start + k for k in range(max_d - n_valid)] + current_indices
                else:
                    row = current_indices[-max_d:]
                target_list.append(row)

        heads_tensor = torch.tensor(heads_indices, device=self.device, dtype=torch.long)
        self.Heads_data = (
            Full_Data[heads_tensor].contiguous().to(torch.float64)
            if len(heads_indices) > 0
            else torch.empty((0, num_cols), device=self.device, dtype=torch.float64)
        )

        def build_tensors(idx_list, max_d):
            if not idx_list:
                return None, None, None, None, None
            T = torch.tensor(idx_list, device=self.device, dtype=torch.long)
            G = Full_Data[T]
            X = G[..., [0, 1, 3]].contiguous().to(torch.float64)
            Y = G[..., 2].unsqueeze(-1).contiguous().to(torch.float64)
            ones = torch.ones_like(G[..., 0]).unsqueeze(-1)
            lat = (G[..., 0] - self.lat_mean_val).unsqueeze(-1)
            dums = G[..., 4:11]
            Locs = torch.cat([ones, lat, dums], dim=-1).contiguous().to(torch.float64)
            is_dummy = (T >= dummy_start).unsqueeze(-1)
            Locs = Locs.masked_fill(is_dummy, 0.0)
            Y = Y.masked_fill(is_dummy, 0.0)
            return X, Y, Locs, T, is_dummy

        self.X_A, self.Y_A, self.Locs_A, self._T_A, self._is_dummy_A = build_tensors(batch_list_A, max_dim_A)
        self.X_AB, self.Y_AB, self.Locs_AB, self._T_AB, self._is_dummy_AB = build_tensors(batch_list_AB, max_dim_AB)
        self.X_ABC, self.Y_ABC, self.Locs_ABC, self._T_ABC, self._is_dummy_ABC = build_tensors(batch_list_ABC, max_dim_ABC)

        self._heads_tensor_stored = heads_tensor if len(heads_indices) > 0 else None
        self._dummy_start_stored = dummy_start
        self._n_real_stored = n_real
        self._n_dummies_stored = n_dummies
        self.n_tails = len(batch_list_A) + len(batch_list_AB) + len(batch_list_ABC)
        self.is_precomputed = True
        return self

## Parameter And Godambe Helpers

In [7]:
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]


def physical_to_log_phi(params):
    sigmasq = float(params["sigmasq"])
    range_lat = float(params["range_lat"])
    range_lon = float(params["range_lon"])
    range_time = float(params["range_time"])
    nugget = float(params["nugget"])
    phi2 = 1.0 / range_lon
    phi1 = sigmasq * phi2
    phi3 = (range_lon / range_lat) ** 2
    phi4 = (range_lon / range_time) ** 2
    return [
        np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
        float(params["advec_lat"]), float(params["advec_lon"]), np.log(nugget),
    ]


def transform_log_phi_to_physical(p):
    phi1, phi2, phi3, phi4 = (torch.exp(p[i]) for i in range(4))
    rlon = 1.0 / phi2
    return torch.stack([
        phi1 / phi2,
        rlon / torch.sqrt(phi3),
        rlon,
        rlon / torch.sqrt(phi4),
        p[4],
        p[5],
        torch.exp(p[6]),
    ])


def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        "sigmasq": np.exp(p[0]) / phi2,
        "range_lat": rlon / phi3 ** 0.5,
        "range_lon": rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat": p[4],
        "advec_lon": p[5],
        "nugget": np.exp(p[6]),
    }


def relative_to_estimate(se_by_key, est_by_key, key, zero_thresh=0.01):
    denom = abs(float(est_by_key[key]))
    if denom >= zero_thresh:
        return float(se_by_key[key] / denom)
    return float(se_by_key[key])


def finite_diff_hessian(nll_fn, p, eps=HESSIAN_EPS):
    n = p.shape[0]
    H = torch.zeros(n, n, device=p.device, dtype=p.dtype)
    for i in range(n):
        p_p = p.detach().clone(); p_m = p.detach().clone()
        p_p[i] += eps; p_m[i] -= eps
        p_p.requires_grad_(True); p_m.requires_grad_(True)
        g_p = torch.autograd.grad(nll_fn(p_p), p_p)[0].detach()
        g_m = torch.autograd.grad(nll_fn(p_m), p_m)[0].detach()
        H[i] = (g_p - g_m) / (2.0 * eps)
    return (H + H.T) / 2.0


def vecchia_per_unit_target_coords(model):
    chunks = []
    if model.Heads_data is not None and model.Heads_data.shape[0] > 0:
        chunks.append(model.Heads_data[:, [0, 1, 3]].to(dtype=DTYPE))
    for X_b in [model.X_A, model.X_AB, model.X_ABC]:
        if X_b is not None and X_b.shape[0] > 0:
            chunks.append(X_b[:, -1, :].to(dtype=DTYPE))
    if not chunks:
        return torch.empty((0, 3), device=DEVICE, dtype=DTYPE)
    return torch.cat(chunks, dim=0)


def make_block_ids(target_coords):
    lat = target_coords[:, 0]
    lon = target_coords[:, 1]
    tim = target_coords[:, 2]
    lat_id = torch.floor((lat - lat.min()) / GODAMBE_BLOCK_LAT_WIDTH).to(torch.long)
    lon_id = torch.floor((lon - lon.min()) / GODAMBE_BLOCK_LON_WIDTH).to(torch.long)
    if GODAMBE_BLOCK_TIME_WIDTH is None or GODAMBE_BLOCK_TIME_WIDTH <= 0:
        time_id = torch.zeros_like(lat_id)
    else:
        time_id = torch.floor((tim - tim.min()) / GODAMBE_BLOCK_TIME_WIDTH).to(torch.long)
    n_lon = int(lon_id.max().item()) + 1 if lon_id.numel() else 1
    n_time = int(time_id.max().item()) + 1 if time_id.numel() else 1
    raw_id = (lat_id * n_lon + lon_id) * n_time + time_id
    _, block_id = torch.unique(raw_id, sorted=True, return_inverse=True)
    return block_id


def score_cov_per_unit_centered(score_mat):
    n_units = score_mat.shape[1]
    score_mean = score_mat.mean(dim=1)
    score_centered = score_mat - score_mean.unsqueeze(1)
    if n_units > 1:
        return score_centered @ score_centered.T / (n_units * (n_units - 1))
    return score_mat @ score_mat.T / max(n_units ** 2, 1)


def score_cov_block_cluster(score_mat, target_coords):
    n_units = score_mat.shape[1]
    scores = score_mat.T.contiguous()
    block_id = make_block_ids(target_coords)
    n_blocks = int(block_id.max().item()) + 1 if block_id.numel() else 0
    block_scores = torch.zeros((n_blocks, scores.shape[1]), device=DEVICE, dtype=DTYPE)
    block_scores.index_add_(0, block_id, scores)
    if n_blocks > 1:
        centered = block_scores - block_scores.mean(dim=0, keepdim=True)
        J = centered.T @ centered * (n_blocks / (n_blocks - 1)) / (n_units ** 2)
    else:
        J = block_scores.T @ block_scores / max(n_units ** 2, 1)
    return J, n_blocks


def compute_vecchia_godambe_real(model, raw_params):
    p_hat = torch.tensor(raw_params[:7], device=DEVICE, dtype=DTYPE, requires_grad=True)

    def nll(p):
        return model.vecchia_batched_likelihood(p)

    H = finite_diff_hessian(nll, p_hat)
    eig = torch.linalg.eigvalsh(H).detach()
    h_abs_min = torch.clamp(torch.min(torch.abs(eig)), min=1e-12)
    h_cond = float((torch.max(torch.abs(eig)) / h_abs_min).detach().cpu())
    beta_hat = model.get_gls_beta(p_hat).detach()

    def per_unit_losses(p):
        return model.vecchia_per_unit_nll_terms(p, beta_hat)

    cols = []
    for k in range(p_hat.shape[0]):
        pp = p_hat.detach().clone(); pm = p_hat.detach().clone()
        pp[k] += SCORE_EPS; pm[k] -= SCORE_EPS
        with torch.no_grad():
            cols.append((per_unit_losses(pp) - per_unit_losses(pm)) / (2.0 * SCORE_EPS))
    score_mat = torch.stack(cols)
    n_units = score_mat.shape[1]
    target_coords = vecchia_per_unit_target_coords(model)
    if target_coords.shape[0] != n_units:
        raise RuntimeError(f"target/score mismatch: targets={target_coords.shape[0]}, scores={n_units}")

    score_mean = score_mat.mean(dim=1)
    p_grad = p_hat.detach().clone().requires_grad_(True)
    profile_grad = torch.autograd.grad(nll(p_grad), p_grad)[0].detach()
    score_grad_diff = profile_grad - score_mean

    J_uncentered = score_mat @ score_mat.T / (n_units ** 2)
    J_centered = score_cov_per_unit_centered(score_mat)
    J_block, n_blocks = score_cov_block_cluster(score_mat, target_coords)
    if GODAMBE_J_METHOD == "block":
        J_main = J_block
    elif GODAMBE_J_METHOD == "per_unit_centered":
        J_main = J_centered
    elif GODAMBE_J_METHOD == "per_unit_uncentered":
        J_main = J_uncentered
    else:
        raise ValueError(f"Unknown GODAMBE_J_METHOD={GODAMBE_J_METHOD!r}")

    eye = torch.eye(H.shape[0], device=DEVICE, dtype=DTYPE)
    h_scale = torch.clamp(torch.mean(torch.abs(torch.diag(H))), min=1.0)
    H_inv = torch.linalg.pinv(H + eye * h_scale * H_RIDGE_SCALE)
    Jac = torch.autograd.functional.jacobian(transform_log_phi_to_physical, p_hat)

    def summarize_J(J):
        G_raw = H_inv @ J @ H_inv
        G_phys = Jac @ G_raw @ Jac.T
        se = torch.sqrt(torch.clamp(torch.diag(G_phys), min=0.0)).detach().cpu().numpy()
        return dict(zip(P_LABELS, [float(x) for x in se]))

    est = backmap_params(raw_params)
    se_main = summarize_J(J_main)
    se_block = summarize_J(J_block)
    se_centered = summarize_J(J_centered)
    se_uncentered = summarize_J(J_uncentered)

    rel_main = {k: relative_to_estimate(se_main, est, k) for k in P_LABELS}
    rel_block = {k: relative_to_estimate(se_block, est, k) for k in P_LABELS}
    rel_centered = {k: relative_to_estimate(se_centered, est, k) for k in P_LABELS}
    rel_uncentered = {k: relative_to_estimate(se_uncentered, est, k) for k in P_LABELS}

    return {
        "gim_j_method": GODAMBE_J_METHOD,
        "gim_n_units": int(n_units),
        "gim_n_blocks": int(n_blocks),
        "gim_h_cond_abs": h_cond,
        "gim_score_mean_max_abs": float(torch.max(torch.abs(score_mean)).detach().cpu()),
        "gim_profile_grad_max_abs": float(torch.max(torch.abs(profile_grad)).detach().cpu()),
        "gim_score_profile_diff_max_abs": float(torch.max(torch.abs(score_grad_diff)).detach().cpu()),
        **{f"gim_se_{k}": v for k, v in se_main.items()},
        **{f"gim_relse_est_{k}": v for k, v in rel_main.items()},
        **{f"gim_relse_est_block_{k}": v for k, v in rel_block.items()},
        **{f"gim_relse_est_centered_{k}": v for k, v in rel_centered.items()},
        **{f"gim_relse_est_uncentered_{k}": v for k, v in rel_uncentered.items()},
    }

## Fit Helpers

In [8]:
def make_params_list(init_physical=INIT_PHYSICAL):
    initial_vals = physical_to_log_phi(init_physical)
    return [Parameter(torch.tensor([val], dtype=DTYPE, device=DEVICE)) for val in initial_vals]


def instantiate_model(spec, daily_hourly_map):
    if spec["kernel"] == "std":
        return kernels_vecchia.fit_vecchia_lbfgs(
            smooth=SMOOTH,
            input_map=daily_hourly_map,
            nns_map=nns_map,
            mm_cond_number=MM_COND_NUMBER,
            nheads=NHEADS,
            limit_A=spec["limit_A"],
            limit_B=spec["limit_B"],
            limit_C=spec["limit_C"],
            daily_stride=DAILY_STRIDE,
        )
    if spec["kernel"] == "hybrid_fresh":
        return fit_vecchia_lbfgs_fresh_hybrid(
            smooth=SMOOTH,
            input_map=daily_hourly_map,
            nns_map=nns_map,
            mm_cond_number=MM_COND_NUMBER,
            nheads=NHEADS,
            limit_A=spec["limit_A"],
            limit_B=spec["lag1_local_count"],
            limit_C=spec["lag2_local_count"],
            daily_stride=DAILY_STRIDE,
            spatial_coords=ordered_base_coords_np,
            lag1_lon_offset=spec["pred_lag1_lon_offset"],
            lag1_fresh_count=spec["lag1_fresh_count"],
            lag2_fresh_count=spec["lag2_fresh_count"],
        )
    raise ValueError(f"unknown kernel {spec['kernel']}")


def fit_one(day_idx, model_key, spec):
    daily_hourly_map = daily_hourly_maps_vecc[day_idx]
    params_list = make_params_list()
    model = instantiate_model(spec, daily_hourly_map)

    t0 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            model.precompute_conditioning_sets()
    else:
        model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.set_optimizer(
        params_list,
        lr=LBFGS_LR,
        max_iter=LBFGS_MAX_EVAL,
        history_size=LBFGS_HISTORY_SIZE,
    )

    t1 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    else:
        out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
    fit_s = time.time() - t1

    godambe = {}
    gim_s = 0.0
    if COMPUTE_GODAMBE:
        t2 = time.time()
        godambe = compute_vecchia_godambe_real(model, out[:7])
        gim_s = time.time() - t2

    est = backmap_params(out)
    valid_count = sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in daily_hourly_map.values())
    row = {
        "year": YEAR,
        "month": MONTH,
        "day_idx": int(day_idx),
        "day": f"{YEAR}-{MONTH:02d}-{day_idx + 1:02d}",
        "model_key": model_key,
        "model": spec["name"],
        "group": spec["group"],
        "kernel": spec["kernel"],
        "allocation": spec["allocation"],
        "limit_A": spec["limit_A"],
        "limit_B": spec["limit_B"],
        "limit_C": spec["limit_C"],
        "lag1_local_count": spec["lag1_local_count"],
        "lag1_fresh_count": spec["lag1_fresh_count"],
        "lag2_local_count": spec["lag2_local_count"],
        "lag2_fresh_count": spec["lag2_fresh_count"],
        "pred_lag1_lon_offset": spec.get("pred_lag1_lon_offset", 0.0),
        "pred_lag2_lon_offset": spec.get("pred_lag2_lon_offset", 0.0),
        "pred_lag1_cells": spec.get("pred_lag1_cells", 0.0),
        "pred_lag2_cells": spec.get("pred_lag2_cells", 0.0),
        "total_conditioning": spec["total_conditioning"],
        "daily_stride": DAILY_STRIDE,
        "n_valid_o3": int(valid_count),
        "loss": float(out[-1]),
        "fit_steps": int(steps_ran),
        "precompute_s": round(precompute_s, 3),
        "fit_s": round(fit_s, 3),
        "gim_s": round(gim_s, 3),
        "total_s": round(precompute_s + fit_s + gim_s, 3),
        **{f"{k}_est": float(v) for k, v in est.items()},
    }
    row["range_t_est"] = row.pop("range_time_est")
    row.update(godambe)
    return row

## Run 5-Day Comparison

In [9]:
records = []

for day_idx in DAYS_LIST:
    print(f"\n{'='*70}\nDay {day_idx + 1}: {YEAR}-{MONTH:02d}-{day_idx + 1:02d}\n{'='*70}")
    for model_key, spec in MODEL_SPECS.items():
        print(f"{spec['name']}: fitting", end="")
        try:
            row = fit_one(day_idx, model_key, spec)
            records.append(row)
            print(f" | loss={row['loss']:.6f} time={row['total_s']:.1f}s advec_lon={row['advec_lon_est']:+.4f}")
        except Exception as exc:
            import traceback
            print(f" | FAILED: {type(exc).__name__}: {exc}")
            traceback.print_exc()
            records.append({
                "year": YEAR,
                "month": MONTH,
                "day_idx": int(day_idx),
                "day": f"{YEAR}-{MONTH:02d}-{day_idx + 1:02d}",
                "model_key": model_key,
                "model": spec["name"],
                "group": spec["group"],
                "kernel": spec["kernel"],
                "error": repr(exc),
            })
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(records)
if SAVE_CSV:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    result_path = OUT_DIR / f"{OUT_PREFIX}_results.csv"
    results_df.to_csv(result_path, index=False)
    print("Saved:", result_path)

display(results_df)


Day 1: 2024-07-01
BaseLocal_A20_B20_C20: fitting | loss=1.345009 time=760.9s advec_lon=-0.1709
NearLocal_L16F02_C12F02_Op0p063: fitting | loss=1.326365 time=367.9s advec_lon=-0.2346
NearLocal_L16F02_C12F02_Op0p126: fitting | loss=1.329963 time=395.8s advec_lon=-0.2389

Day 2: 2024-07-02
BaseLocal_A20_B20_C20: fitting | loss=1.526875 time=848.5s advec_lon=-0.1980
NearLocal_L16F02_C12F02_Op0p063: fitting | loss=1.490885 time=461.3s advec_lon=-0.2550
NearLocal_L16F02_C12F02_Op0p126: fitting | loss=1.504150 time=464.1s advec_lon=-0.2548

Day 3: 2024-07-03
BaseLocal_A20_B20_C20: fitting | loss=1.510502 time=698.2s advec_lon=-0.2476
NearLocal_L16F02_C12F02_Op0p063: fitting | loss=1.458949 time=520.6s advec_lon=-0.2709
NearLocal_L16F02_C12F02_Op0p126: fitting | loss=1.475608 time=435.3s advec_lon=-0.2705

Day 4: 2024-07-04
BaseLocal_A20_B20_C20: fitting | loss=1.433092 time=949.6s advec_lon=-0.1460
NearLocal_L16F02_C12F02_Op0p063: fitting | loss=1.395137 time=530.5s advec_lon=-0.1547
NearLoc

Traceback (most recent call last):
  File "/var/folders/9p/53hd4c7d2fl193h4jwp194wc0000gn/T/ipykernel_23614/379001434.py", line 8, in <module>
    row = fit_one(day_idx, model_key, spec)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/9p/53hd4c7d2fl193h4jwp194wc0000gn/T/ipykernel_23614/970045541.py", line 61, in fit_one
    out, steps_ran = model.fit_vecc_lbfgs(params_list, optimizer, max_steps=LBFGS_MAX_STEPS, grad_tol=GRAD_TOL)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/joonwonlee/Documents/GEMS_TCO-1/src/GEMS_TCO/kernels_vecchia.py", line 514, in fit_vecc_lbfgs
    loss = optimizer.step(closure)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/gems_gpu/lib/python3.12/site-packages/torch/optim/optimizer.py", line 487, in wrapper
    out = func(*args, **kwargs)
          ^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/gems_gpu/lib/python3.12/site-packages/torch/utils/_c

 | loss=1.416295 time=470.1s advec_lon=-0.2383
NearLocal_L16F02_C12F02_Op0p126: fitting | loss=1.433347 time=500.0s advec_lon=-0.2438
Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_5day_nearlocal_vs_base_050226_results.csv


,year,month,day_idx,day,model_key,model,group,kernel,allocation,limit_A,...,gim_relse_est_centered_advec_lon,gim_relse_est_centered_nugget,gim_relse_est_uncentered_sigmasq,gim_relse_est_uncentered_range_lat,gim_relse_est_uncentered_range_lon,gim_relse_est_uncentered_range_time,gim_relse_est_uncentered_advec_lat,gim_relse_est_uncentered_advec_lon,gim_relse_est_uncentered_nugget,error
0,2024,7,0,2024-07-01,local_baseline::BaseLocal_A20_B20_C20,BaseLocal_A20_B20_C20,local_baseline,std,A20; B=same+local20; C=same+local20,20.0,...,0.047445,0.028799,0.012516,0.021338,0.021258,0.026066,0.004437,0.047445,0.028799,NaN
1,2024,7,0,2024-07-01,near_local_hybrid::NearLocal_L16F02_C12F02_Op0...,NearLocal_L16F02_C12F02_Op0p063,near_local_hybrid,hybrid_fresh,A20; B=same+local16+fresh2; C=same+local12+fre...,20.0,...,0.023686,0.029872,0.013725,0.024189,0.022813,0.022109,0.148105,0.023686,0.029872,NaN
2,2024,7,0,2024-07-01,near_local_hybrid::NearLocal_L16F02_C12F02_Op0...,NearLocal_L16F02_C12F02_Op0p126,near_local_hybrid,hybrid_fresh,A20; B=same+local16+fresh2; C=same+local12+fre...,20.0,...,0.021892,0.026520,0.013786,0.027545,0.022819,0.021884,0.123205,0.021891,0.026520,NaN
3,2024,7,1,2024-07-02,local_baseline::BaseLocal_A20_B20_C20,BaseLocal_A20_B20_C20,local_baseline,std,A20; B=same+local20; C=same+local20,20.0,...,0.039675,0.055193,0.016797,0.027348,0.029277,0.031274,0.151870,0.039675,0.055193,NaN
4,2024,7,1,2024-07-02,near_local_hybrid::NearLocal_L16F02_C12F02_Op0...,NearLocal_L16F02_C12F02_Op0p063,near_local_hybrid,hybrid_fresh,A20; B=same+local16+fresh2; C=same+local12+fre...,20.0,...,0.013538,0.053291,0.018088,0.034675,0.032978,0.028242,0.092945,0.013538,0.053291,NaN
5,2024,7,1,2024-07-02,near_local_hybrid::NearLocal_L16F02_C12F02_Op0...,NearLocal_L16F02_C12F02_Op0p126,near_local_hybrid,hybrid_fresh,A20; B=same+local16+fresh2; C=same+local12+fre...,20.0,...,0.012694,0.056182,0.017939,0.040779,0.035034,0.027854,0.076635,0.012694,0.056182,NaN
6,2024,7,2,2024-07-03,local_baseline::BaseLocal_A20_B20_C20,BaseLocal_A20_B20_C20,local_baseline,std,A20; B=same+local20; C=same+local20,20.0,...,0.026373,0.027738,0.012164,0.022005,0.021789,0.029856,0.073944,0.026373,0.027738,NaN
7,2024,7,2,2024-07-03,near_local_hybrid::NearLocal_L16F02_C12F02_Op0...,NearLocal_L16F02_C12F02_Op0p063,near_local_hybrid,hybrid_fresh,A20; B=same+local16+fresh2; C=same+local12+fre...,20.0,...,0.012632,0.047627,0.015610,0.036895,0.037831,0.032947,0.043052,0.012632,0.047627,NaN
8,2024,7,2,2024-07-03,near_local_hybrid::NearLocal_L16F02_C12F02_Op0...,NearLocal_L16F02_C12F02_Op0p126,near_local_hybrid,hybrid_fresh,A20; B=same+local16+fresh2; C=same+local12+fre...,20.0,...,0.010523,0.051980,0.015086,0.041135,0.041878,0.033576,0.037576,0.010523,0.051980,NaN
9,2024,7,3,2024-07-04,local_baseline::BaseLocal_A20_B20_C20,BaseLocal_A20_B20_C20,local_baseline,std,A20; B=same+local20; C=same+local20,20.0,...,0.022872,0.026864,0.013051,0.022074,0.022099,0.025261,0.126796,0.022872,0.026864,NaN


## Compare Loss, Parameters, And Godambe SE

In [10]:
if "error" in results_df.columns:
    ok_df = results_df[results_df["error"].isna()].copy() if results_df["error"].notna().any() else results_df.copy()
else:
    ok_df = results_df.copy()

summary_cols = [
    "loss", "total_s", "fit_s", "gim_s",
    "sigmasq_est", "range_lat_est", "range_lon_est", "range_t_est",
    "advec_lat_est", "advec_lon_est", "nugget_est",
]
if COMPUTE_GODAMBE:
    summary_cols += [
        "gim_se_range_lon", "gim_se_range_time", "gim_se_advec_lon", "gim_se_nugget",
        "gim_relse_est_range_lon", "gim_relse_est_range_time", "gim_relse_est_advec_lon", "gim_relse_est_nugget",
        "gim_h_cond_abs", "gim_n_blocks",
    ]
summary_cols = [c for c in summary_cols if c in ok_df.columns]

agg_spec = {f"{c}_mean": (c, "mean") for c in summary_cols}
agg_spec.update({f"{c}_median": (c, "median") for c in summary_cols})
model_summary = (
    ok_df
    .groupby(["model", "group", "kernel", "total_conditioning", "pred_lag1_lon_offset"], sort=False)
    .agg(n_days=("day", "nunique"), **agg_spec)
    .reset_index()
)

day_loss = ok_df.pivot_table(index="day", columns="model", values="loss", aggfunc="first")
base_col = "BaseLocal_A20_B20_C20"
if base_col in day_loss.columns:
    for col in list(day_loss.columns):
        if col != base_col:
            day_loss[f"delta_{col}_minus_base"] = day_loss[col] - day_loss[base_col]

print("Model-level summary:")
display(model_summary)
print("Day-level loss comparison:")
display(day_loss)

if SAVE_CSV:
    summary_path = OUT_DIR / f"{OUT_PREFIX}_summary.csv"
    day_loss_path = OUT_DIR / f"{OUT_PREFIX}_day_loss.csv"
    model_summary.to_csv(summary_path, index=False)
    day_loss.to_csv(day_loss_path)
    print("Saved:", summary_path, day_loss_path)

Model-level summary:


,model,group,kernel,total_conditioning,pred_lag1_lon_offset,n_days,loss_mean,total_s_mean,fit_s_mean,gim_s_mean,...,gim_se_range_lon_median,gim_se_range_time_median,gim_se_advec_lon_median,gim_se_nugget_median,gim_relse_est_range_lon_median,gim_relse_est_range_time_median,gim_relse_est_advec_lon_median,gim_relse_est_nugget_median,gim_h_cond_abs_median,gim_n_blocks_median
0,BaseLocal_A20_B20_C20,local_baseline,std,62.0,0.000,4,1.453870,814.31025,511.05275,301.80025,...,0.023597,0.126990,0.012586,0.129996,0.060272,0.058501,0.058194,0.065864,191.008740,600.5
1,NearLocal_L16F02_C12F02_Op0p063,near_local_hybrid,hybrid_fresh,54.0,0.063,5,1.417526,470.06520,288.31780,174.89540,...,0.020766,0.112657,0.006495,0.114695,0.061317,0.057900,0.032717,0.063258,256.867235,601.0
2,NearLocal_L16F02_C12F02_Op0p126,near_local_hybrid,hybrid_fresh,54.0,0.126,5,1.431575,461.06140,279.03680,175.21340,...,0.023854,0.114388,0.005993,0.120051,0.064818,0.057455,0.033759,0.060038,260.669534,601.0


Day-level loss comparison:


model,BaseLocal_A20_B20_C20,NearLocal_L16F02_C12F02_Op0p063,NearLocal_L16F02_C12F02_Op0p126,delta_NearLocal_L16F02_C12F02_Op0p063_minus_base,delta_NearLocal_L16F02_C12F02_Op0p126_minus_base
day,,,,,
2024-07-01,1.345009,1.326365,1.329963,-0.018644,-0.015046
2024-07-02,1.526875,1.490885,1.504150,-0.035990,-0.022725
2024-07-03,1.510502,1.458949,1.475608,-0.051553,-0.034894
2024-07-04,1.433092,1.395137,1.414804,-0.037955,-0.018287
2024-07-05,NaN,1.416295,1.433347,NaN,NaN


Saved: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_5day_nearlocal_vs_base_050226_summary.csv /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log/real_vecc_2024_5day_nearlocal_vs_base_050226_day_loss.csv


## Reading Guide

Use this as a real-data model comparison, not a truth-known simulation:

- Lower `loss` is better profile likelihood under the same data and same initial values.
- `gim_se_*` gives physical-scale Godambe SE estimates.
- `gim_relse_est_*` is `SE / |estimate|` when the estimate is not close to zero; for near-zero advection parameters it falls back to absolute SE.
- `gim_h_cond_abs` flags unstable Hessian inversions; very large values should be treated as a warning.
- `delta_*_minus_base` in `day_loss` is the cleanest first check: negative means the hybrid beats the original BaseLocal on that day.

If the NearLocal models lower loss without inflating Godambe SE or producing unstable Hessians, they are strong candidates for the final real-data strategy.